# Normality tests and outliers 
## Introduction


## Variability of Gaussian samples


In [ ]:
import warnings
# Just to avoid messages not useful for the book
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(111)  # For reproducibility, now we know the trick

# Sampling parameters
mean_temp = 36.8
std_temp = 0.4
sample_sizes = [12, 130]
num_cols = 4

# Pre-generate random values in a dictionnary, with sample size
# as the key, and the distribution object as the value.
# It's super Pythonic, and we could have done the same differently
random_samples = {
    size: np.random.normal(
        loc=mean_temp, scale=std_temp, size=(num_cols, size))
    for size in sample_sizes}

# Create subplots and share x-axis for consistent scaling
fig, axes = plt.subplots(
    nrows=len(sample_sizes),
    ncols=num_cols,
    figsize=(6, 3),
    sharex=True)

# Main title for the entire figure
fig.suptitle("Gaussian distribution samples")

for row, size in enumerate(sample_sizes):
    for col in range(num_cols):
        ax = axes[row, col]  # Get the current subplot
        # We will plot kernel density estimates (KDE) for 'smoothing'
        # the distribution of the observation data points
        sns.histplot(
            random_samples[size][col],
            stat="density",
            binwidth=0.1,
            kde=True,
            ax=ax,
            label=f"n={size}")
        # Plot the KDE separately with desired color and linewidth
        sns.kdeplot(
            random_samples[size][col],
            ax=ax,
            color='orangered',
            linewidth=2)
        ax.set_xlabel('Temperature (°C)')
        # Only label the y-axis for the first plot in each row
        ax.set_ylabel('Frequency' if col == 0 else "")

        # Add legend only to the last plot in each row
        if col == num_cols-1:
            ax.legend()

plt.xlim((35, 38))
sns.despine()
plt.tight_layout();

In [ ]:

np.random.seed(111)

# Parameters
mean_temp = 36.8
std_temp = 0.4
labels = list("ABCDEFGHIJ")
sample_size = 12

# Generate and reshape data
data = np.random.normal(
    loc=mean_temp,
    scale=std_temp,
    size=len(labels) * sample_size)
data = data.reshape((len(labels), sample_size))

# Create a list of sample labels for each data point
sample_labels = np.repeat(labels, sample_size)  # Copy each label 12 times

# Flatten the data array
flattened_data = data.flatten()

# Create a dictionary for seaborn's data format
data_dict = {
    'Sample': sample_labels,
    'Value': flattened_data}

# Plot using seaborn with the dictionary data format
plt.figure(figsize=(3.5, 3))
sns.swarmplot(
    x='Sample', y='Value', data=data_dict,
    color="olivedrab",
    marker='s')

plt.title("Gaussian samples");

## Assessing the normality of data
### Q-Q plots
#### Q-Q plots with SciPy


In [ ]:

import scipy.stats as st

np.random.seed(111)

# Sample Generation (more descriptive variable names)
normal_data = np.random.normal(size=100)
exponential_data = np.random.exponential(size=100)

# Let's present the distributions with their title in a dictionary
distributions_dict = {
    "Normal distribution": normal_data,
    "Exponential distribution": exponential_data}

# Plotting (using a loop for conciseness and consistency)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

for i, (title, data) in enumerate(distributions_dict.items()):
    st.probplot(data, plot=axes[i])
    axes[i].set_title(title)
plt.tight_layout();

#### Q-Q plot with Pingouin


In [ ]:

import pingouin as pg

np.random.seed(111)

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

for i, (title, data) in enumerate(distributions_dict.items()):
    pg.qqplot(
        data,
        dist='norm', # compare the data against the normal distribution
        ax=axes[i],
        confidence=0.95, # Add 95% confidence intervals
    )
    axes[i].set_title(title)
plt.tight_layout();

#### Alternative options


In [ ]:
from scipy.stats import norm

def custom_qq_plot(data, ax=None, **kwargs):
    """Creates a Q-Q plot using Matplotlib.

    Args:
        data: the data to plot.
        ax: the axes to plot on. If None, a new figure and axes is created.
        **kwargs: Additional keyword arguments passed to ax.scatter for customization

    Returns:
        matplotlib.axes.Axes: the axes object on which the plot was drawn.
    """
    if ax is None:
        _, ax = plt.subplots()

    # Sort the data
    data_sorted = np.sort(data)

    # Theoretical quantiles
    theoretical_quantiles = norm.ppf(np.linspace(0, 1, len(data)))

    # Scatter plot of sample quantiles vs. theoretical quantiles
    ax.scatter(
        theoretical_quantiles,
        data_sorted,
        **kwargs)

    # Add diagonal line
    lims = [
        np.min([ax.get_xlim(), ax.get_ylim()]),
        np.max([ax.get_xlim(), ax.get_ylim()])]
    ax.plot(
        lims, lims,
        'r--', alpha=0.75, zorder=0)

    # Labels and title
    ax.set_xlabel("Theoretical quantiles")
    ax.set_ylabel("Sample quantiles")
    ax.set_title("Q-Q plot")
    ax.legend()

    return ax

In [ ]:

np.random.seed(111)

# Standardize both datasets
normal_data_std = (normal_data - np.mean(normal_data)) / np.std(normal_data)
exponential_data_std = (
    exponential_data - np.mean(exponential_data)) / np.std(exponential_data)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))

custom_qq_plot(
    normal_data_std,
    ax=axes[0],
    label='Normal')

custom_qq_plot(
    exponential_data_std,
    ax=axes[1],
    label='Exponential')

plt.tight_layout();

### Statistical tests for normality


#### D'Agostino-Pearson omnibus K² normality test
##### Skewness and kurtosis 


In [ ]:

from scipy.stats import skewnorm

# Generate data for skewed distributions
positive_skew_data = skewnorm.rvs(a=10, loc=10, scale=5, size=1000)
negative_skew_data = skewnorm.rvs(a=-10, loc=10, scale=5, size=1000)

# Generate data for kurtosis examples
normal_kurtosis_data = np.random.normal(loc=0, scale=1, size=1000)
# The Laplace distribution has higher kurtosis than normal
high_kurtosis_data = np.random.laplace(loc=0, scale=1/np.sqrt(2),size=1000)
# The uniform distribution has lower kurtosis
low_kurtosis_data = np.random.uniform(-np.sqrt(3), np.sqrt(3), size=1000)

# Create the figure and subplots
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

# Left subplot: Skewness
sns.kdeplot(
    positive_skew_data, bw_adjust=2, ax=axes[0], 
    color='skyblue', lw=2, label='Positive')
sns.kdeplot(
    negative_skew_data, bw_adjust=2, ax=axes[0], 
    color='salmon', lw=2, label='Negative')
axes[0].set_title('Skewness')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].legend(loc='upper right', fontsize=8)

# Right subplot: Kurtosis
sns.kdeplot(
    normal_kurtosis_data, bw_adjust=2, ax=axes[1], color='lightgreen', 
    lw=2, label='Mesokurtic')
sns.kdeplot(
    high_kurtosis_data, bw_adjust=2, ax=axes[1], color='gold', lw=2, 
    label='Leptokurtic')
sns.kdeplot(
    low_kurtosis_data, bw_adjust=2, ax=axes[1], color='lightcoral', 
    lw=2, label='Platykurtic')
axes[1].set_title('Kurtosis')
axes[1].set_xlabel('Value')
axes[1].set_xlim((-4, 4))
axes[1].set_ylabel('Frequency')
axes[1].legend(loc='upper left', fontsize=8)

plt.tight_layout();

In [ ]:
np.random.seed(111)

# Calculate and print results
for title, data in distributions_dict.items():
    print(
        title, '\n'
        " skewness:", st.skew(data).round(3),
        " kurtosis:", st.kurtosis(data, fisher=False).round(3))

In [ ]:
# Calculate descriptive statistics using `describe`
print("Normal samples:")
print("Skewness =", st.describe(normal_data).skewness.round(3))
print("Kurtosis (excess) =", st.describe(normal_data).kurtosis.round(3))

##### How the K² test works


##### Manual implementation


In [ ]:
from scipy.stats import skew, kurtosis, chi2

def dagostino_pearson_test(data):
    """Performs the D'Agostino-Pearson omnibus K2 test for normality.

    Args:
        data (array-like): the data to test for normality.

    Returns:
        statistic: K2 test statistic.
        p_value: P value for the hypothesis test.
    """
    n = len(data)
    if n < 8:
        raise ValueError("Sample size must be at least 8 for the test.")

    # Calculate sample skewness and kurtosis (Fisher's definition)
    s = skew(data)
    k = kurtosis(data, fisher=True)  

    # Calculate intermediate terms
    y = s * np.sqrt((n + 1) * (n + 3) / (6 * (n - 2)))
    beta_two = (
        (3 * (n**2 + 27*n - 70) * (n + 1) * (n + 3))
        /
        ((n - 2) * (n + 5) * (n + 7) * (n + 9))
    )
    w_two = -1 + np.sqrt(2 * (beta_two - 1))
    delta = 1 / np.sqrt(np.log(np.sqrt(w_two)))
    alpha = np.sqrt(2 / (w_two - 1))
    
    # Calculate Z-scores
    z_skew = delta * np.log(y / alpha + np.sqrt((y / alpha)**2 + 1))
    z_kurt = (k / (2 * np.sqrt(3 * (n - 1) / ((n + 1) * (n + 3)))))

    # Calculate K2 statistic
    k2 = z_skew**2 + z_kurt**2

    # Calculate P value using chi-square distribution 
    # with 2 degrees of freedom
    p_value = 1 - chi2.cdf(k2, 2)

    return k2, p_value

# Use the function of the previously defined distributions
print("D'Agostino-Pearson test:")

for title, data in distributions_dict.items():
    k2, pval = dagostino_pearson_test(data)
    print(
        f"{title.removesuffix('distribution'):<12} \
K² = {k2:.2f}, P value = {pval:.3f}")

##### K² test with SciPy


In [ ]:
# Function to perform the test and print results
def dagostino_pearson_test_and_print(data, title):
    k2, pval = st.normaltest(data)
    print(f"{title:<12} K² = {k2:.2f}; P value = {pval:.3f}")

# Perform tests and print results using the previous distributions
print("D'Agostino-Pearson Omnibus K² test (SciPy):")

for title, data in distributions_dict.items():
    dagostino_pearson_test_and_print(
        data, title.removesuffix('distribution'))

#### Shapiro-Wilk test
##### How the Shapiro-Wilk test works
##### Shapiro-Wilk test with SciPy


In [ ]:
# Function to perform the Shapiro-Wilk test and print results
def shapiro_wilk_test_and_print(data, title):
    statistic, p_value = st.shapiro(data)
    print(f"{title:<12} W = {statistic:.3f}, P value = {p_value:.3f}")
    # print(f"{' ':<15} {'(Reject H0 if p < 0.05)':>30}")
    # H0: Data is normal

# Perform Shapiro-Wilk test and print results
print("Shapiro-Wilk test (SciPy):")

for title, data in distributions_dict.items():
    shapiro_wilk_test_and_print(data, title.removesuffix('distribution'))

#### Normality tests using Pingouin


In [ ]:
import pingouin as pg

# Function to perform tests and print results
def pingouin_normality_tests(data, title):
    print("Normality tests (Pingouin):", title)
    print(f"{'Test':<20} {'W':<8} {'P value':<6}")

    # Shapiro-Wilk test
    shapiro_results = pg.normality(data, method='shapiro')
    print(f"{'Shapiro-Wilk':<20} {shapiro_results['W'][0]:<8.3f} \
{shapiro_results['pval'][0]:<6.3f}")

    # D'Agostino-Pearson test
    dagostino_results = pg.normality(data, method='normaltest')
    print(f"{'D\'Agostino-Pearson':<20} {dagostino_results['W'][0]:<8.3f} \
{dagostino_results['pval'][0]:<6.3f}")

    print("-" * 40)

# Perform tests and print results for previous distributions
for title, data in distributions_dict.items():
    pingouin_normality_tests(data, title.removesuffix('distribution'))

#### Omnibus test for normality using statsmodels


In [ ]:
import statsmodels.api as sm

# Function to perform the omnibus test and print results
def omnibus_normality_test(data, title):
    omnibus_stat, omnibus_pvalue = sm.stats.stattools.omni_normtest(data)

    print("Omnibus normality test (statstmodels):", title)
    print(f" Omnibus statistic = {omnibus_stat:.3f}")
    print(f" Omnibus P value = {omnibus_pvalue:.3f}")
    # print(f"{'(Reject H0 if p < 0.05)':>30}")  # H0: Data is normal

# Perform tests and print results for the previous distributions
for title, data in distributions_dict.items():
    omnibus_normality_test(data, title.removesuffix('distribution'))

#### Kolmogorov-Smirnov test


In [ ]:

np.random.seed(111)  # for reproducibility

# Generation of a bimodal distribution
sample_size = 1000
sample_bimodal = (
    np.random.randn(sample_size)
    * 0.5
    + 1
    - 2 * (np.random.random(sample_size) > 0.5)
)

# ECDF Function
def ecdf(data):
    """Compute ECDF for a one-dimensional array of values."""
    x = np.sort(data)
    y = np.arange(1, len(data) + 1) / len(data)
    return x, y

# Calculate ECDFs
sample_x, sample_ecdf = ecdf(sample_bimodal)
normal_cdf = norm.cdf(sample_x)

# KS Statistic
ks_stat = np.max(np.abs(normal_cdf - sample_ecdf))

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(6, 3))

# Histogram and KDE Plot
sns.histplot(
    sample_bimodal,
    label="Sampled data (bimodal)",
    stat="density",
    kde=True,
    ax=axes[0])
sns.lineplot(
    x=sample_x,
    y=norm.pdf(sample_x),
    color="xkcd:orange",
    label="Expected (normal)",
    ax=axes[0])
axes[0].set_title(
    "Histogram and KDE",)
axes[0].legend(fontsize=9)

# ECDF Plot
sns.lineplot(
    x=sample_x,
    y=sample_ecdf,
    label="Sampled data (bimodal)",
    ax=axes[1])
sns.lineplot(
    x=sample_x,
    y=normal_cdf,
    color="xkcd:orange",
    label="Expected (normal)",
    ax=axes[1])

# Highlight the max vertical difference (KS statistic)
max_diff_idx = np.argmax(
    np.abs(normal_cdf - sample_ecdf))
axes[1].vlines(
    x=sample_x[max_diff_idx],
    ymin=min(sample_ecdf[max_diff_idx], normal_cdf[max_diff_idx]),
    ymax=max(sample_ecdf[max_diff_idx], normal_cdf[max_diff_idx]),
    colors="black",
    linestyles="dashed",
    label=f"KS statistic: {ks_stat:.3f}")

axes[1].set_title('ECDF')
axes[1].legend(fontsize=9, loc='upper left');

In [ ]:
# Perform the one-sample Kolmogorov-Smirnov test
ks_statistic, p_value_ks = st.kstest(
    sample_bimodal,
    'norm', # 'norm' specifies the normal distribution
)

# Output the results with clear formatting
print("One-sample Kolmogorov-Smirnov test (SciPy)")
print("Test statistic (D) =", ks_statistic.round(3))
print("P value =", p_value_ks.round(6))
if p_value_ks < 0.05:  # Typical significance level
    print("Reject H0: evidence suggests the sample is NOT \
normally distributed")
else:
    print("Fail to reject H0: insufficient evidence to conclude the sample\
 is not normally distributed")

#### Additional normality tests


## Outliers
### Graphical approaches for outlier detection
#### General-purpose outlier detection


#### The case of log-normal data 


In [ ]:

np.random.seed(111)

# Parameters
mean = np.log(80) # Mean of the underlying normal distribution 
sigma = 0.8       # Standard deviation of the underlying normal distribution
labels = list("ABCDEFGHIJ")
sample_size = 15

# Generate data
data = np.random.lognormal(
    mean, sigma, size=(len(labels), sample_size))

# Flatten and create labels for seaborn
flattened_data = data.flatten()
sample_labels = np.repeat(labels, sample_size)
data_dict = {
    'Sample': sample_labels,
    'Value': flattened_data}

# Plotting
fig, axes = plt.subplots(figsize=(6,3), nrows=1, ncols=2,)

# Plot 1: Linear Scale
sns.swarmplot(
    x=np.tile(labels, sample_size),
    y=data.flatten(),
    color="crimson",
    ax=axes[0])
axes[0].set_title("Linear scale")

# Plot 2: Log Scale
sns.swarmplot(
    x=np.tile(labels, sample_size),
    y=data.flatten(),
    color="steelblue",
    ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("Log scale")

plt.tight_layout();

#### Cook's distance and leverage
##### Plotting the Cook's distance


In [ ]:
import pandas as pd

# Load the dataset
iris = sm.datasets.get_rdataset("iris").data

# Rename columns to avoid PatsyError
iris.columns = iris.columns.str.replace(".", "_")

# Create one exagerated outlier
iris.loc[50, 'Petal_Length'] = (
    iris['Petal_Length'].mean() + 3 * iris['Petal_Length'].std())

# Look at the structure of the dataframe
print(iris.info())

In [ ]:

from statsmodels.formula.api import ols
from statsmodels.stats.outliers_influence import OLSInfluence

# fit the regression model using statsmodels library
# (see chapter on linear regression for more details)
model = ols(
    formula="Petal_Length ~ Petal_Width",
    data=iris).fit()

# Calculate Cooks' distance, leverage, DFFITS, and standardized residuals
influence = OLSInfluence(model)

cooks_distance = influence.cooks_distance[0]
leverage = influence.hat_matrix_diag
dffits = influence.dffits[0]
standardized_residuals = influence.resid_studentized_internal
predicted_values = model.fittedvalues

# Create a figure with four subplots sharing a y-axis
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(6, 6))

# Subplot 1: Regression Plot
sns.scatterplot(
    x='Petal_Width', y='Petal_Length', data=iris,
    hue=cooks_distance, size=cooks_distance,
    alpha=.8, edgecolor='black', linewidth=1,
    ax=axes[0, 0])
sns.regplot(
    x='Petal_Width', y='Petal_Length', data=iris,
    scatter=False, ci=False,
    ax=axes[0, 0],
    line_kws={'color': 'darkgrey', 'ls': '--'})
axes[0, 0].set_xlabel("Petal width")
axes[0, 0].set_ylabel("Petal length")
axes[0, 0].set_title('Regression plot')
axes[0, 0].legend(
    title="Cooks' distance", fontsize=8, loc='lower left')


# Subplot 2: Cooks' distance vs. Predicted Values
sns.scatterplot(
    x=predicted_values, y=cooks_distance,
    color='orangered', marker='s', alpha=.75,
    ax=axes[0, 1])
axes[0, 1].set_title("Influence plot")
axes[0, 1].set_xlabel("Predicted values (petal length)")
axes[0, 1].set_ylabel("Cook's distance")


# Subplot 3: DFFITS Plot
axes[1, 0].stem(dffits)
axes[1, 0].set_title('DFFITS plot')
axes[1, 0].set_xlabel('Observation index')
axes[1, 0].set_ylabel('DFFITS')

# Subplot 4: Influence Plot (Bubble Plot)
influence_data = pd.DataFrame({
    'Leverage': leverage,
    'Std residuals': standardized_residuals,
    'Cooks distance': cooks_distance})

sns.scatterplot(
    x='Leverage', y='Std residuals', data=influence_data,
    size='Cooks distance', hue='Cooks distance',
    alpha=.8, palette='crest', ax=axes[1,1])
axes[1, 1].set_title('Leverage-residual plot')
axes[1, 1].legend(fontsize=8)

# Display all plots
plt.tight_layout();

##### Interpreting Cook's distance


### Statistical outlier detection
#### Grubbs test


In [ ]:
def grubbs_gmax(n, alpha=0.05, onesided=False):
    """
    Critical value for Grubbs test for outlier.

    >>> grubbs_gmax(n=8, alpha=0.05, onesided=False)
    2.126645087195626
    >>> grubbs_gmax(n=8, alpha=0.05, onesided=True)
    2.031652001549952
    See also:
    https://ycopin.pages.in2p3.fr/Informatique-Python/Annales/22-grubbs/exam22_corrige.html
    """

    sl = alpha / n    # one-sided
    if not onesided:  # two-sided
        sl /= 2

    # Critical value (squared) of the t distribution w/ n-2 DoF and
    # significance level sl
    cv2 = st.t.isf(sl, n - 2) ** 2
    gmax = (n - 1) * np.sqrt(cv2 / (n - 2 + cv2) / n)

    return gmax

In [ ]:
# Generate normal sample with an outlier
np.random.seed(123)

sample_size = 100
sample_grubbs = np.random.normal(0, 1, sample_size)
sample_grubbs[0] = 5  # Introduce an exaggerated outlier

# Grubbs Test (manual implementation)
def grubbs_test_manual(data, alpha=0.05):
    """Performs a two-sided Grubbs test manually."""

    sample_mean = np.mean(data)
    sample_std = np.std(data, ddof=1)

    # Calculate G statistic (using absolute value for two-sided test)
    G = np.max(np.abs(data - sample_mean)) / sample_std

    # Critical value from Grubbs table or using t-distribution
    G_critical = grubbs_gmax(n=len(data), alpha=alpha, onesided=False)

    return G, G_critical

# Perform manual Grubbs test
G_calculated, G_critical = grubbs_test_manual(sample_grubbs)

print("Grubbs test (manual):")
print(f" Test Statistic (G) = {G_calculated:.3f}")
print(f" Critical Value (G) = {G_critical:.3f}")
if G_calculated > G_critical:
    print("Reject H0: evidence suggests the presence \
of a significant outlier.")
else:
    print("Fail to reject H0: insufficient evidence \
to conclude there is an outlier.")

#### Grubbs outlier detection with scikit-posthocs


In [ ]:
import scikit_posthocs as ph

def find_outliers(sample):
    """
    Identifies and returns outliers from a dataset using Grubb's test.

    Args:
        sample: a NumPy array of data points.

    Returns:
        A NumPy array containing the outlier(s) detected by Grubb's test.
    """
    
    # Apply Grubb's test to filter outliers
    filtered = ph.outliers_grubbs(sample)
    
    # Find the difference between the original and filtered arrays
    outliers = np.setdiff1d(sample, filtered)
    
    return outliers

# Example with the previous sample array
outliers = find_outliers(sample_grubbs)
print("Outlier(s) detected:", outliers)

#### Additional statistical methods for outlier detection


In [ ]:
# Returns a boolean array indicating whether each point is an outlier
print("MAD-median resulting array (Pingouin):")
print(pg.madmedianrule(sample_grubbs))

## Conclusion
